# Sets

**Topic:** `3.3` &nbsp;|&nbsp; **Module 3: Core Data Structures**

*Reference implementations for all 10 exercises plus the robotics challenge.*

> *Course:* Python Master Course &nbsp;|&nbsp; **Project:** ROBO-X (M3)

### Navigate

* [Lesson](lesson.ipynb) &nbsp;|&nbsp; [Exercises](exercises.ipynb) &nbsp;|&nbsp; [Solutions](solution.ipynb) &nbsp;|&nbsp; [Research](research.ipynb) &nbsp;|&nbsp; [Quiz](quiz.ipynb) &nbsp;|&nbsp; [Mini-project](mini_project.ipynb) &nbsp;|&nbsp; [Robotics challenge](robotics_challenge.ipynb)

Module index: [03_data_structures/README.md](../README.md)

> **Compare, do not copy**
>
> Work each exercise yourself first. When you open this notebook, compare your approach with the reference: check the reasoning, the edge cases and the complexity. If your solution differs, that is fine — verify it with the tests provided and keep whichever is clearer.

## Solution 1 — Deduplicate sensor tags with a set

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** set construction, deduplication, sorted output

**Explanation and reasoning**

set(tags) collapses duplicates in one O(n) pass because uniqueness is the type's construction semantics, not a filter applied afterward. sorted() then restores the deterministic order the set never stored - required for logs and snapshot tests.

In [ ]:
def unique_tags(tags):
    """Distinct tags, sorted; input untouched."""
    return sorted(set(tags))

**Complexity**

Time O(n log n) dominated by the sort; space O(n).

**Edge cases and failure modes**

Empty input returns []. All-duplicates input yields one element.

**Alternative approaches**

dict.fromkeys(tags) keeps first-seen order without a sort - a useful alternative to name.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert unique_tags(['warn', 'fault', 'warn']) == ['fault', 'warn']
src = ['a', 'a']
unique_tags(src)
assert src == ['a', 'a']

---

## Solution 2 — Check whether all required keys were supplied

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** subset, difference, validation

**Explanation and reasoning**

The difference `required - supplied` IS the definition of missing, so the boolean and the report come from one computation - there is no separate loop that could disagree with it. Converting both sides also means callers may pass lists or sets interchangeably.

In [ ]:
def check_keys(supplied, required):
    """(ok, missing) - ok iff every required key is present."""
    required, supplied = set(required), set(supplied)
    missing = sorted(required - supplied)
    return (not missing), missing

**Complexity**

Time O(n + m) expected; space O(n + m).

**Edge cases and failure modes**

Exact match gives (True, []). Extra supplied keys never fail. Empty required passes.

**Alternative approaches**

required <= supplied is the boolean in one operator, but then missing needs the difference anyway.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert check_keys(['host'], ['host', 'port']) == (False, ['port'])
assert check_keys(['a', 'b'], ['a']) == (True, [])
assert check_keys([], []) == (True, [])

---

## Solution 3 — Find codes present in both subsystems

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** intersection, set conversion, sorted

**Explanation and reasoning**

Intersection is exactly the 'present in both' relation, computed by walking the smaller operand against a hash table of the larger - linear rather than the quadratic pair of nested membership loops it replaces. Sorting happens once, at the end, because set order is not stored.

In [ ]:
def common_faults(a, b):
    """Sorted fault codes reported by both subsystems."""
    return sorted(set(a) & set(b))

**Complexity**

Time O((n + m) + k log k) where k is the overlap size; space O(n + m).

**Edge cases and failure modes**

Disjoint inputs return []. Duplicates within an input never duplicate output.

**Alternative approaches**

sorted(c for c in a if c in set(b)) works but scans and rebuilds without deduping a.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert common_faults(['E1', 'E2'], ['E2', 'E3']) == ['E2']
assert common_faults(['E1'], ['E2']) == []

---

## Solution 4 — Remove every occurrence of blacklisted values

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** difference, list comprehension with membership, order preservation

**Explanation and reasoning**

Converting the blacklist once upgrades every membership test from a linear scan to a hash lookup, so the whole filter is O(n + m) instead of O(n * m). The comprehension walks the input in order, so sequence is preserved - something set difference alone could not promise.

In [ ]:
def strip_blacklist(values, blacklist):
    """Filter blacklisted values, preserving order."""
    blocked = set(blacklist)          # one O(m) conversion
    return [v for v in values if v not in blocked]

**Complexity**

Time O(n + m) expected; space O(m) for the set plus the output.

**Edge cases and failure modes**

Empty blacklist returns a copy of the input. Empty input returns [].

**Alternative approaches**

list(set(values) - blocked) dedupes too, which changes semantics - order and duplicates are lost.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert strip_blacklist([1, 2, 3, 4], [2, 4]) == [1, 3]
src = [1, 2]
strip_blacklist(src, [1])
assert src == [1, 2]

---

## Solution 5 — Detect duplicate frames in O(n)

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** seen set, linear scan, order preservation

**Explanation and reasoning**

One set answers presence, two lists record the two outcomes in encounter order - the partition invariant len(unique) + len(duplicates) == len(frames) holds by construction because every frame takes exactly one branch.

In [ ]:
def find_duplicates(frames):
    """(unique in first-seen order, later repeats in order seen)."""
    seen = set()
    unique, duplicates = [], []
    for frame in frames:
        if frame in seen:
            duplicates.append(frame)
        else:
            seen.add(frame)
            unique.append(frame)
    return unique, duplicates

**Complexity**

Time O(n) expected; space O(n).

**Edge cases and failure modes**

No duplicates gives ([], duplicates). Triplicates append twice. Empty input gives ([], []).

**Alternative approaches**

collections.Counter(frames) then filtering counts > 1 loses encounter order of repeats.

**Tests — run the cell to verify the reference solution**

In [ ]:
u, d = find_duplicates([('a', 1), ('b', 2), ('a', 1)])
assert u == [('a', 1), ('b', 2)] and d == [('a', 1)]
u, d = find_duplicates([])
assert u == [] and d == []

---

## Solution 6 — Compute a coverage report with set algebra

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** union, difference, coverage ratio

**Explanation and reasoning**

Three algebra expressions answer the three report questions from one pair of sets. max(len(planned), 1) is the guard that keeps an empty plan from dividing by zero - and it deliberately reports 0.0 rather than claiming full coverage of nothing.

In [ ]:
def coverage(planned, executed):
    """missing, unexpected and executed-coverage ratio."""
    planned, executed = set(planned), set(executed)
    shared = len(planned & executed)
    ratio = round(shared / max(len(planned), 1), 2)
    return {
        'missing': sorted(planned - executed),
        'unexpected': sorted(executed - planned),
        'ratio': ratio,
    }

**Complexity**

Time O(n + m + k log k); space O(n + m).

**Edge cases and failure modes**

Empty planned gives ratio 0.0. Exact match gives ([], [], 1.0).

**Alternative approaches**

Manual counting loops would rebuild what &, - already express in one readable operator each.

**Tests — run the cell to verify the reference solution**

In [ ]:
out = coverage(['w1', 'w2'], ['w2', 'w3'])
assert out == {'missing': ['w1'], 'unexpected': ['w3'], 'ratio': 0.5}
assert coverage([], [])['ratio'] == 0.0
assert coverage(['a'], ['a'])['ratio'] == 1.0

---

## Solution 7 — Group anagram signatures into sets

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** set signatures, default grouping, sorted keys

**Explanation and reasoning**

A frozenset of the word's letters is immutable and hashable, so it can be a dict key - the exact role a plain set cannot play. setdefault keeps grouping to one line, and the final dict comprehension sorts each group so the report is reproducible despite set-keyed iteration order.

In [ ]:
def group_anagrams(words):
    """frozenset of letters -> sorted words sharing that signature."""
    groups = {}
    for word in words:
        signature = frozenset(word)      # hashable group key
        groups.setdefault(signature, []).append(word)
    return {sig: sorted(ws) for sig, ws in groups.items()}

**Complexity**

Time O(total letters * log word length) from sorting groups; space O(total letters).

**Edge cases and failure modes**

Empty input gives {}. Single word maps to a one-element list.

**Alternative approaches**

tuple(sorted(word)) as key handles repeated letters more precisely than a frozenset - name this trade-off.

**Tests — run the cell to verify the reference solution**

In [ ]:
out = group_anagrams(['bat', 'tab', 'cat'])
assert out[frozenset('abt')] == ['bat', 'tab']
assert out[frozenset('act')] == ['cat']
assert group_anagrams([]) == {}

---

## Solution 8 — Find the first value present in exactly one of two streams

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** symmetric difference, order recovery, index mapping

**Explanation and reasoning**

The set equality test first collapses the 'no exclusives at all' case - including equal sets in different orders - into one line. The scans then respect arrival order within each stream, with stream a checked first so ties break to a exactly as specified; membership in the opposite set is O(1) per step.

In [ ]:
def first_exclusive(a, b):
    """Earliest value present in exactly one stream; ties -> a."""
    sa, sb = set(a), set(b)
    if sa == sb:
        return None
    for value in a:
        if value not in sb:
            return value
    for value in b:
        if value not in sa:
            return value
    return None   # unreachable when sa != sb

**Complexity**

Time O(n + m) expected; space O(n + m).

**Edge cases and failure modes**

Equal sets return None. Only-b exclusives fall to the second loop. Empty inputs handled by the equality test.

**Alternative approaches**

sorted(a - b) picks the smallest value, not the earliest - a different and wrong order semantics.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert first_exclusive([1, 2, 3], [3, 4]) == 1
assert first_exclusive([5, 6], [6, 5]) is None
assert first_exclusive([], [9]) == 9

---

## Solution 9 — Maintain a rolling window of distinct values

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** windowing, set per slice, counting distinct

**Explanation and reasoning**

The range stop `len(values) - size + 1` guarantees every slice is full, so no partial window is ever counted; when size exceeds the data the range is empty and the result is []. Each window's set collapses repeats, making len() the distinct count.

In [ ]:
def distinct_counts(values, size):
    """Distinct-element count per sliding window."""
    if size < 1:
        raise ValueError('size must be at least 1')
    return [
        len(set(values[i:i + size]))
        for i in range(len(values) - size + 1)
        if i + size <= len(values)
    ]

**Complexity**

Time O(n * size) for n windows; space O(size) per window plus the output.

**Edge cases and failure modes**

size > len(values) returns []. size == 1 gives all ones. All-identical windows count 1.

**Alternative approaches**

A counter dict updated incrementally is O(n) total but far more code; sets keep it obvious.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert distinct_counts([1, 2, 1, 3], 2) == [2, 2, 2]
assert distinct_counts([1, 2], 5) == []
assert distinct_counts([7, 7, 7], 1) == [1, 1, 1]

---

## Solution 10 — Implement set algebra without the operators

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** set semantics, membership loops, complexity reasoning

**Explanation and reasoning**

Rebuilding the algebra shows what the operators hide: union is two insertions into a fresh set (uniqueness falls out of the set itself), intersection and difference are single filters with O(1) membership probes against the other side. The complexity matches the built-ins because the built-ins are hash-table walks too.

In [ ]:
def my_union(a, b):
    out = set()
    for value in a:
        out.add(value)
    for value in b:
        out.add(value)
    return out


def my_intersection(a, b):
    return {v for v in a if v in b}


def my_difference(a, b):
    return {v for v in a if v not in b}

**Complexity**

Union O(n + m); intersection/difference O(n) expected.

**Edge cases and failure modes**

Disjoint union merges fully. Empty operands yield the other set (union) or set() (the rest).

**Alternative approaches**

Looping both sides for intersection would be O(n * m) - the hash probe is the entire point.

**Tests — run the cell to verify the reference solution**

In [ ]:
for x, y in [({1, 2}, {2, 3}), (set(), {1}), ({1}, set())]:
    assert my_union(x, y) == x | y
    assert my_intersection(x, y) == x & y
    assert my_difference(x, y) == x - y
    assert x == x and y == y   # inputs untouched

---

# Robotics challenge solution — ROBO-X Challenge: Fault Supervisor Core

**Explanation**

Fault state is a set because presence is binary and queries are hot. Clearing maps to discard (idempotent - clearing an inactive code is not an error), and the criticality question is one intersection whose truthiness becomes the halt decision - the kind of expression sets make trivial to test.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / 'shared').is_dir()), None)
if _root is not None:
    sys.path.insert(0, str(_root))

from shared.robo_x_sim import SimulatedRobot  # noqa: E402

CRITICAL = {'E_STOP', 'E_BATT'}


def supervise(robot, reports):
    """Fold fault reports into active state; flag criticals."""
    active = set()
    for code in reports:
        if code.startswith('C'):
            active.discard(code[1:])     # clear directive
        else:
            active.add(code)
    breached = active & CRITICAL
    return {'active': sorted(active),
            'critical': sorted(breached),
            'halt': bool(breached)}

**Complexity**

Time O(r + c) for r reports and c critical codes; space O(active faults).

**Notes and trade-offs**

Clearing an absent code is a no-op. No active faults give halt False with empty lists.

In [ ]:
robot = SimulatedRobot(name='rover-01', battery_wh=48.0)
out = supervise(robot, ['E_BATT', 'E1', 'CE_BATT'])
assert out['active'] == ['E1']
assert out['halt'] is False
assert supervise(robot, ['E_STOP'])['halt'] is True

## Reflection

For each exercise where your solution differed from the reference, write one sentence explaining *why* yours was better or worse. This is the highest-value part of the notebook:

- A case the reference handles but mine did not.
- A case mine handles better — and what the reference is missing.
- A performance difference that matters at robot scale.